In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import csv
from urllib.parse import urljoin
import re

# **Выбор источника**
Есть три сайта, предоставляющие доступ к базам данных по сборкам кубика Рубика.

* https://reco.nz/   
* https://www.cuberoot.me/
* https://speedcubedb.com/ - раздел с базой данных удалён 4 года назад.

Из этих сайтов только на https://reco.nz/ есть время выполнения каждого этапа сборки, что необходимо для данного проекта.

In [ ]:
BASE_URL = "https://reco.nz/"

def get_solve_ids_from_main_page():
  """Парсить главную страницу, чтобы найти ID сборок"""
  ids = []
  for page in range(2,3):
    url = urljoin(BASE_URL, f"/?sort=id_desc&3x3=on&CFOP=on&page={page}")
    resp = requests.get(url)
    soup = BeautifulSoup(resp.text, 'html.parser')

    table = soup.find('table')
    if not table:
        print("Table not found on main page. Check selector.")
        return []


    for row in table.find_all('tr')[1:]:  # заголовок не берем
        first_cell = row.find('td')
        if first_cell:
            solve_id = first_cell.get_text(strip=True)
            if solve_id.isdigit():
                ids.append(solve_id)
  return ids

In [ ]:
def fetch_solve_details(solve_id):
    """
    Получить всю необходимую информацию о сборке.
    Возвращаемое значение - словарь со следующими полями:
    ['solve_id', 'solver', 'solve_time', 'pll', 'pll_time', 'Split', 'STM', 'STPS', 'ETM', 'ETPS']
    """
    detail_url = urljoin(BASE_URL, f"/solve/{solve_id}")
    response = requests.get(detail_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Извлекаем имя и время из заголовка h1
    h1_text = soup.find('h1').get_text(strip=True)
    # Формат: "Yiheng Wang - 3.85 3x3 solve"
    match = re.match(r'(.+?)\s*-\s*([\d.]+)\s', h1_text)
    if match:
        name = match.group(1).strip()
        solve_time = match.group(2)
    else:
        name = None
        solve_time = None

    # Извлекаем строку с PLL или EPLL
    # Ищем в тексте страницы строки, содержащие "// PLL" или "// EPLL"
    content_text = soup.get_text()
    pll_epll = None
    for line in content_text.split('\n'):
        line = line.strip()
        if '// PLL' in line:
            pll_epll = line.split('// PLL')[0].strip()
            break
        if '// EPLL' in line:
            pll_epll = line.split('// EPLL')[0].strip()
            break

    # Извлекаем данные из последнего столбца таблицы (столбец PLL)
    # Находим таблицу по заголовкам
    table = soup.find('table')
    if table:
        rows = table.find_all('tr')
        # Ищем строку с заголовками, чтобы понять структуру
        headers = []
        header_row = rows[0] if rows else None
        if header_row:
            headers = [th.get_text(strip=True) for th in header_row.find_all(['th', 'td'])]

        # Ищем строку с данными (обычно вторая строка после заголовков)
        data = {}
        for row in rows[1:]:
            cols = row.find_all(['td', 'th'])
            if len(cols) == len(headers):
                for i, header in enumerate(headers):
                    data[header] = cols[i].get_text(strip=True)

        # Извлекаем значения из последнего столбца (PLL)
        pll_column_index = len(headers) - 1
        last_col_data = {}
        if len(rows) > 1:
            data_row = rows[1].find_all(['td', 'th'])
            if len(data_row) == len(headers):
                for i, header in enumerate(headers):
                    last_col_data[header] = data_row[i].get_text(strip=True)

        # Оставляем только нужные метрики из последнего столбца
        result = {
            'solve_id': solve_id,
            'solver': name,
            'solve_time': solve_time,
            'pll': pll_epll,
        }

        # Добавляем данные из последнего столбца
        last_col_name = headers[pll_column_index] if headers else 'PLL'
        if pll_column_index < len(data_row):
            result['pll_time'] = data_row[pll_column_index].get_text(strip=True) if pll_column_index < len(data_row) else None

        # На самом деле, структура таблицы: строки - метрики, столбцы - этапы
        # Нужно транспонировать данные
        if headers and len(rows) > 1:
            metrics = []
            for row in rows[1:]:
                cols = row.find_all(['td', 'th'])
                if cols:
                    metrics.append([col.get_text(strip=True) for col in cols])

            # metrics[0] - pll_time, metrics[1] - Split, metrics[2] - STM,
            # metrics[3] - STPS, metrics[4] - ETM, metrics[5] - ETPS

            metric_names = ['pll_time', 'Split', 'STM', 'STPS', 'ETM', 'ETPS']
            for i, metric_row in enumerate(metrics):
                if i < len(metric_names) and len(metric_row) > pll_column_index:
                    result[metric_names[i]] = metric_row[pll_column_index]

        return result

    return None


In [ ]:
print("Finding solve IDs on the main page")
solve_ids = get_solve_ids_from_main_page()
print(f"Found {len(solve_ids)} solves.")

# Ограничение количества данных для тестовых запусков
solve_ids = solve_ids[:5]

results = []
for sid in solve_ids:
    print(f"Processing solve {sid}")
    try:
        details = fetch_solve_details(sid)
        results.append(details)
    except Exception as e:
        print(f"  Error: {e}")
        results.append({'solve_id': sid, 'solver': None, 'solve_time': None, 'pll': None,'pll_time':None, 'Split':None, 'STM':None, 'STPS':None, 'ETM':None, 'ETPS':None})
    time.sleep(1)  # чтобы не перегрузить сервер

with open('rubik_solves.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['solve_id', 'solver', 'solve_time', 'pll', 'pll_time', 'Split', 'STM', 'STPS', 'ETM', 'ETPS'])
    writer.writeheader()
    writer.writerows(results)

print(f"\nDone. Saved {len(results)} solves to rubik_solves.csv")

Finding solve IDs on the main page
Found 50 solves.
Processing solve 12926
Processing solve 12925
Processing solve 12924
Processing solve 12923
Processing solve 12922

Done. Saved 5 solves to rubik_solves.csv
